# Treccani validation of Cultura

## Settings

In [1]:
from pathlib import Path
import re

import duckdb
import pandas as pd

ROOT = Path.cwd().parent
AI_ANSWERS = ROOT / "annotations" / "treccani_validation" / "treccani_ai_annotations_cultura289.parquet"
DATABASE = ROOT / "data" / "cultura" / "humans_clean_v2_sample_treccani.duckdb"

## Treccani answers found by the AI

In [2]:
def window_years(answer):
    years = []
    for year, decade in re.findall(r"\b(\d{3,4})(s?)\b", answer or ""):
        years += [int(year), int(year) + 9] if decade else [int(year)]
    if years:
        return min(years), max(years)
    centuries = [int(c) for c in re.findall(r"(\d{1,2})(?:st|nd|rd|th) century", answer or "")]
    if centuries:
        return (min(centuries) - 1) * 100 + 1, max(centuries) * 100
    return None, None


raw = pd.read_parquet(AI_ANSWERS)
treccani = pd.DataFrame({
    "qid": raw["entity"].map(lambda entity: entity["qid"]),
    "name": raw["entity"].map(lambda entity: entity["label_en"]),
    "treccani_polity": raw["polity_ai_matched"].map(lambda answer: answer["answer"]),
    "treccani_polity_id": raw["polity_ai_matched"].map(lambda answer: int(answer["source_verbatim"][0].split("|")[0]) if len(answer["source_verbatim"]) else None),
    "treccani_window": raw["productivity_window_ai_extracted"].map(lambda answer: answer["answer"]),
})
treccani[["treccani_start", "treccani_end"]] = treccani["treccani_window"].map(window_years).tolist()
assert treccani["qid"].is_unique
print(len(treccani), "individuals")
treccani

50 individuals


,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end
0,Q16577209,Maurizio Cattaneo,Republic of Genoa,589.0,1451-1484,1451,1484
1,Q109861078,Olimpio Cozzolino,Kingdom of Italy,350.0,1894-1937,1894,1937
2,Q3607061,Aicone,Kingdom of Italy,350.0,905-915,905,915
3,Q318593,Girolamo Fracastoro,Republic of Venice,397.0,before 1502 to 1553,1502,1553
4,Q26997486,Bartolo da San Gimignano,Holy Roman Empire Minor States,783.0,1293,1293,1293
5,Q18758541,Domenico Forges Davanzati,Kingdom of Naples (Napoleonic),1140.0,1768-1809,1768,1809
6,Q15428551,Biagio Buonaccorsi,Republic of Florence,768.0,1498-1512,1498,1512
7,Q95387776,Carbone Malocello,Republic of Genoa,589.0,1216-1235,1216,1235
8,Q1399161,Fazio of Uberti,Holy Roman Empire,1612.0,from the 1330s to the late 1350s,1330,1359
9,Q53004,Vittorio De Sica,Republic of Italy,1428.0,1923-1974,1923,1974


## Cultura window and polities

In [3]:
connection = duckdb.connect(str(DATABASE), read_only=True)
qids = treccani["qid"].tolist()

windows = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           individual_enriched.peak_productivity.start_year AS cultura_start,
           individual_enriched.peak_productivity.end_year AS cultura_end
    FROM individual_enriched
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
""", [qids]).df()
windows["cultura_window"] = windows.apply(lambda row: None if pd.isna(row["cultura_start"]) else f"{int(row['cultura_start'])}-{int(row['cultura_end'])}", axis=1)

sub_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(polity_match.polity.cliopatria_id) AS cultura_sub_polity_ids,
           string_agg(polity_match.polity.name, ' / ') AS cultura_sub_polities
    FROM individual_enriched, unnest(individual_enriched.polity) AS unnested(polity_match)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
    GROUP BY qid
""", [qids]).df()

meta_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(DISTINCT meta.cliopatria_id) AS cultura_meta_polity_ids,
           string_agg(DISTINCT meta.name, ' / ') AS cultura_meta_polities
    FROM individual_enriched,
         unnest(individual_enriched.polity) AS unnested(polity_match),
         unnest(polity_match.polity.meta_polities) AS unnested_meta(meta)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
      AND meta.start_year <= individual_enriched.peak_productivity.end_year
      AND meta.end_year >= individual_enriched.peak_productivity.start_year
    GROUP BY qid
""", [qids]).df()

cultura = windows.merge(sub_polities, on="qid", how="left").merge(meta_polities, on="qid", how="left")
print(len(cultura), "individuals,", cultura["cultura_sub_polities"].notna().sum(), "with a polity")
cultura

50 individuals, 50 with a polity


,qid,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q26997486,1261,1274,1261-1274,[768],Republic of Florence,[1612],Holy Roman Empire
1,Q109861078,1898,1922,1898-1922,[350],Kingdom of Italy,NaN,NaN
2,Q3607061,904,918,904-918,"[350, 500]",Kingdom of Italy / Kingdom of Lower Burgundy,[1611],Carolingian Empire
3,Q318593,1507,1528,1507-1528,[397],Republic of Venice,NaN,NaN
4,Q3627864,852,871,852-871,[422],Duchy of Naples,NaN,NaN
5,Q50330355,1598,1604,1598-1604,[1043],Iberian Union,[1622],Spanish Empire
6,Q3436971,1448,1463,1448-1463,[923],Duchy of Milan,[1612],Holy Roman Empire
7,Q97974535,1300,1322,1300-1322,[768],Republic of Florence,[1612],Holy Roman Empire
8,Q27553577,1270,1294,1270-1294,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire
9,Q26246988,1238,1246,1238-1246,[523],Holy Roman Empire,[1612],Holy Roman Empire


## Comparison table

In [4]:
comparison = treccani.merge(cultura, on="qid", how="left")
assert len(comparison) == len(treccani)
comparison

,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q16577209,Maurizio Cattaneo,Republic of Genoa,589.0,1451-1484,1451,1484,1431,1446,1431-1446,[589],Republic of Genoa,NaN,NaN
1,Q109861078,Olimpio Cozzolino,Kingdom of Italy,350.0,1894-1937,1894,1937,1898,1922,1898-1922,[350],Kingdom of Italy,NaN,NaN
2,Q3607061,Aicone,Kingdom of Italy,350.0,905-915,905,915,904,918,904-918,"[350, 500]",Kingdom of Italy / Kingdom of Lower Burgundy,[1611],Carolingian Empire
3,Q318593,Girolamo Fracastoro,Republic of Venice,397.0,before 1502 to 1553,1502,1553,1507,1528,1507-1528,[397],Republic of Venice,NaN,NaN
4,Q26997486,Bartolo da San Gimignano,Holy Roman Empire Minor States,783.0,1293,1293,1293,1261,1274,1261-1274,[768],Republic of Florence,[1612],Holy Roman Empire
5,Q18758541,Domenico Forges Davanzati,Kingdom of Naples (Napoleonic),1140.0,1768-1809,1768,1809,1777,1788,1777-1788,"[1140, 1000]",Kingdom of Naples (Napoleonic) / Kingdom of Spain,[1622],Spanish Empire
6,Q15428551,Biagio Buonaccorsi,Republic of Florence,768.0,1498-1512,1498,1512,1496,1526,1496-1526,[768],Republic of Florence,NaN,NaN
7,Q95387776,Carbone Malocello,Republic of Genoa,589.0,1216-1235,1216,1235,1223,1235,1223-1235,[589],Republic of Genoa,NaN,NaN
8,Q1399161,Fazio of Uberti,Holy Roman Empire,1612.0,from the 1330s to the late 1350s,1330,1359,1400,1422,1400-1422,"[397, 783]",Republic of Venice / Holy Roman Empire Minor S...,[1612],Holy Roman Empire
9,Q53004,Vittorio De Sica,Republic of Italy,1428.0,1923-1974,1923,1974,1940,1974,1940-1974,[1428],Republic of Italy,NaN,NaN


## Polity accuracy

In [5]:
as_list = lambda ids: [] if ids is None or isinstance(ids, float) else list(ids)
names = lambda text: [] if not isinstance(text, str) else [name.strip() for name in text.split(" / ")]
sub_ids = comparison["cultura_sub_polity_ids"].map(as_list)
meta_ids = comparison["cultura_meta_polity_ids"].map(as_list)
sub_names = comparison["cultura_sub_polities"].map(names)
meta_names = comparison["cultura_meta_polities"].map(names)
comparison["polity_match"] = [
    "sub polity" if treccani_id in subs or treccani_name in sub_name_list else
    "meta polity" if treccani_id in metas or treccani_name in meta_name_list else "none"
    for treccani_id, treccani_name, subs, metas, sub_name_list, meta_name_list
    in zip(comparison["treccani_polity_id"], comparison["treccani_polity"], sub_ids, meta_ids, sub_names, meta_names)
]
comparison["polity_ok"] = comparison["polity_match"] != "none"
comparison["has_treccani_polity"] = comparison["treccani_polity_id"].notna()
print((~comparison["has_treccani_polity"]).sum(), "individual(s) without a Treccani polity, left out of the polity accuracy")
comparison[["name", "treccani_polity", "cultura_sub_polities", "cultura_meta_polities", "polity_match", "polity_ok"]]

1 individual(s) without a Treccani polity, left out of the polity accuracy


,name,treccani_polity,cultura_sub_polities,cultura_meta_polities,polity_match,polity_ok
0,Maurizio Cattaneo,Republic of Genoa,Republic of Genoa,NaN,sub polity,True
1,Olimpio Cozzolino,Kingdom of Italy,Kingdom of Italy,NaN,sub polity,True
2,Aicone,Kingdom of Italy,Kingdom of Italy / Kingdom of Lower Burgundy,Carolingian Empire,sub polity,True
3,Girolamo Fracastoro,Republic of Venice,Republic of Venice,NaN,sub polity,True
4,Bartolo da San Gimignano,Holy Roman Empire Minor States,Republic of Florence,Holy Roman Empire,none,False
5,Domenico Forges Davanzati,Kingdom of Naples (Napoleonic),Kingdom of Naples (Napoleonic) / Kingdom of Spain,Spanish Empire,sub polity,True
6,Biagio Buonaccorsi,Republic of Florence,Republic of Florence,NaN,sub polity,True
7,Carbone Malocello,Republic of Genoa,Republic of Genoa,NaN,sub polity,True
8,Fazio of Uberti,Holy Roman Empire,Republic of Venice / Holy Roman Empire Minor S...,Holy Roman Empire,meta polity,True
9,Vittorio De Sica,Republic of Italy,Republic of Italy,NaN,sub polity,True


## Productivity window accuracy

In [6]:
overlap = (comparison[["treccani_end", "cultura_end"]].min(axis=1, skipna=False) - comparison[["treccani_start", "cultura_start"]].max(axis=1, skipna=False) + 1).clip(lower=0)
treccani_span = comparison["treccani_end"] - comparison["treccani_start"] + 1
comparison["overlap_years"] = overlap
comparison["overlap_share"] = (overlap / treccani_span).round(2)
comparison["window_ok"] = comparison["overlap_years"].fillna(0) > 0
comparison["has_treccani_window"] = comparison["treccani_start"].notna()
print((~comparison["has_treccani_window"]).sum(), "individual(s) without a Treccani window, left out of the window accuracy")
print(comparison["cultura_start"].isna().sum(), "individual(s) without a Cultura window, counted as wrong")
comparison[["name", "treccani_window", "cultura_window", "overlap_years", "overlap_share", "window_ok"]]

0 individual(s) without a Treccani window, left out of the window accuracy
0 individual(s) without a Cultura window, counted as wrong


,name,treccani_window,cultura_window,overlap_years,overlap_share,window_ok
0,Maurizio Cattaneo,1451-1484,1431-1446,0,0.00,False
1,Olimpio Cozzolino,1894-1937,1898-1922,25,0.57,True
2,Aicone,905-915,904-918,11,1.00,True
3,Girolamo Fracastoro,before 1502 to 1553,1507-1528,22,0.42,True
4,Bartolo da San Gimignano,1293,1261-1274,0,0.00,False
5,Domenico Forges Davanzati,1768-1809,1777-1788,12,0.29,True
6,Biagio Buonaccorsi,1498-1512,1496-1526,15,1.00,True
7,Carbone Malocello,1216-1235,1223-1235,13,0.65,True
8,Fazio of Uberti,from the 1330s to the late 1350s,1400-1422,0,0.00,False
9,Vittorio De Sica,1923-1974,1940-1974,35,0.67,True


## Accuracy of Cultura against Treccani

In [7]:
polity_rows = comparison[comparison["has_treccani_polity"]]
window_rows = comparison[comparison["has_treccani_window"]]
pd.DataFrame({
    "field": ["polity", "productivity window"],
    "correct": [polity_rows["polity_ok"].sum(), window_rows["window_ok"].sum()],
    "individuals": [len(polity_rows), len(window_rows)],
    "accuracy": [polity_rows["polity_ok"].mean().round(2), window_rows["window_ok"].mean().round(2)],
})

,field,correct,individuals,accuracy
0,polity,37,49,0.76
1,productivity window,38,50,0.76
